# SCL CP8D 0.8 µm NMOS Characterization

This notebook analyses NMOS transfer curves at several fixed `VDS` values and an output-characteristic sweep. It estimates `gm`, `gds`, `ro`, `gm/ID`, and intrinsic gain where the data supports a matched bias point.

**Model caveat:** Ngspice 42 reported ignored HSPICE-specific parameters for this model library. Treat extracted results as preliminary until model compatibility is validated. Keep the PDK model files and licensed/proprietary materials out of public GitHub unless redistribution is explicitly permitted.

## Inputs

Place the raw `.dat` files listed below in `../data/raw/`:
- `scl_nmos_transfer_vds_0p5.dat`, `_1.dat`, `_2.dat`, `_3.dat`, `_4.dat`, `_5.dat`
- `scl_nmos_output_fixed.dat`

The code assumes `wrdata` output has the swept voltage in column 2 (index 1) and current in the last column, consistent with the multi-sweep files inspected during this project.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Set this to the directory containing the raw Ngspice .dat files.
DATA_DIR = Path("../data/raw")
OUT_TABLES = Path("../results/tables")
OUT_FIGURES = Path("../results/figures")
OUT_TABLES.mkdir(parents=True, exist_ok=True)
OUT_FIGURES.mkdir(parents=True, exist_ok=True)

def load_wrdata(path, x_col=1, current_col=-1):
    """Load Ngspice wrdata output; return sorted x values and |current| in A."""
    data = np.loadtxt(path)
    x = data[:, x_col]
    current = np.abs(data[:, current_col])
    order = np.argsort(x)
    return x[order], current[order]

In [ ]:
transfer_files = {
    0.5: "scl_nmos_transfer_vds_0p5.dat",
    1.0: "scl_nmos_transfer_vds_1.dat",
    2.0: "scl_nmos_transfer_vds_2.dat",
    3.0: "scl_nmos_transfer_vds_3.dat",
    4.0: "scl_nmos_transfer_vds_4.dat",
    5.0: "scl_nmos_transfer_vds_5.dat",
}

transfer = {}
rows = []

for vds, filename in transfer_files.items():
    path = DATA_DIR / filename
    if not path.exists():
        print(f"Missing transfer file: {path}")
        continue
    vgs, id_a = load_wrdata(path, x_col=1)
    gm = np.gradient(id_a, vgs)
    gm_id = np.divide(gm, id_a, out=np.full_like(gm, np.nan), where=id_a > 1e-12)
    transfer[vds] = {"vgs": vgs, "id": id_a, "gm": gm, "gm_id": gm_id}
    for i in range(len(vgs)):
        rows.append({
            "VDS_V": vds, "VGS_V": vgs[i], "ID_A": id_a[i],
            "ID_uA": id_a[i]*1e6, "gm_S": gm[i], "gm_uS": gm[i]*1e6,
            "gm_ID_per_V": gm_id[i]
        })

if not transfer:
    raise FileNotFoundError("No NMOS transfer files loaded. Check DATA_DIR and filenames.")

transfer_df = pd.DataFrame(rows)
transfer_df.to_csv(OUT_TABLES / "scl08_nmos_transfer_parameters.csv", index=False)
display(transfer_df.iloc[::10].round(4))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
for vds, d in transfer.items():
    ax.plot(d["vgs"], d["id"]*1e6, label=f"VDS = {vds:g} V")
ax.set(xlabel=r"$V_{GS}$ (V)", ylabel=r"$I_D$ ($\mu$A)",
       title="SCL 0.8 µm NMOS Transfer Characteristics")
ax.grid(True); ax.legend(); fig.tight_layout()
fig.savefig(OUT_FIGURES / "scl08_nmos_transfer.png", dpi=200)
plt.show()

fig, ax = plt.subplots(figsize=(9, 6))
for vds, d in transfer.items():
    ax.plot(d["vgs"], d["gm"]*1e6, label=f"VDS = {vds:g} V")
ax.set(xlabel=r"$V_{GS}$ (V)", ylabel=r"$g_m$ ($\mu$S)",
       title="NMOS Transconductance")
ax.grid(True); ax.legend(); fig.tight_layout()
fig.savefig(OUT_FIGURES / "scl08_nmos_gm.png", dpi=200)
plt.show()

fig, ax = plt.subplots(figsize=(9, 6))
for vds, d in transfer.items():
    ax.plot(d["vgs"], d["gm_id"], label=f"VDS = {vds:g} V")
ax.set(xlabel=r"$V_{GS}$ (V)", ylabel=r"$g_m/I_D$ (V$^{-1}$)",
       title="NMOS Transconductance Efficiency")
ax.set_ylim(bottom=0); ax.grid(True); ax.legend(); fig.tight_layout()
fig.savefig(OUT_FIGURES / "scl08_nmos_gm_id.png", dpi=200)
plt.show()

## Output characteristics and small-signal output parameters

The output sweep is expected to contain ten consecutive blocks for `VGS = 0.5, 1.0, …, 5.0 V`, with 101 `VDS` points per block. Verify that row ordering matches the original Ngspice sweep before interpreting results.

In [ ]:
output_path = DATA_DIR / "scl_nmos_output_fixed.dat"
if not output_path.exists():
    raise FileNotFoundError(f"Missing output sweep: {output_path}")

out = np.loadtxt(output_path)
n_vgs, n_vds = 10, 101
if out.shape[0] != n_vgs*n_vds:
    raise ValueError(f"Expected {n_vgs*n_vds} rows; found {out.shape[0]}.")

# The first column is the swept VDS in the output-sweep file.
vds_matrix = out[:, 0].reshape(n_vgs, n_vds)
id_matrix = np.abs(out[:, -1]).reshape(n_vgs, n_vds)
vds_values = vds_matrix[0]
vgs_values = np.arange(0.5, 5.01, 0.5)

if not np.allclose(vds_matrix, vds_values[None, :]):
    raise ValueError("VDS sweep blocks do not match; inspect raw data ordering.")

fig, ax = plt.subplots(figsize=(9, 6))
for i, vgs in enumerate(vgs_values):
    ax.plot(vds_values, id_matrix[i]*1e6, label=f"VGS = {vgs:g} V")
ax.set(xlabel=r"$V_{DS}$ (V)", ylabel=r"$I_D$ ($\mu$A)",
       title="SCL 0.8 µm NMOS Output Characteristics")
ax.grid(True); ax.legend(ncol=2); fig.tight_layout()
fig.savefig(OUT_FIGURES / "scl08_nmos_output.png", dpi=200)
plt.show()

gds_matrix = np.gradient(id_matrix, vds_values, axis=1)
ro_matrix = np.divide(1, gds_matrix, out=np.full_like(gds_matrix, np.nan),
                      where=gds_matrix > 0)

In [ ]:
# Match transfer-derived gm with output-derived ID and gds at VDS = 2, 3, 4 V.
matched_rows = []
for vgs in [1.0, 2.0, 3.0, 4.0, 5.0]:
    i = np.argmin(np.abs(vgs_values-vgs))
    for target_vds in [2.0, 3.0, 4.0]:
        j = np.argmin(np.abs(vds_values-target_vds))
        if target_vds not in transfer:
            continue
        tr = transfer[target_vds]
        gm = np.interp(vgs, tr["vgs"], tr["gm"])
        current = id_matrix[i, j]
        gds = gds_matrix[i, j]
        ro = 1/gds if gds > 0 else np.nan
        matched_rows.append({
            "VGS_V": vgs, "VDS_V": vds_values[j], "ID_uA": current*1e6,
            "gm_uS": gm*1e6, "gds_uS": gds*1e6,
            "ro_kOhm": ro/1e3,
            "gm_ID_per_V": gm/current if current > 1e-12 else np.nan,
            "intrinsic_gain_V_per_V": gm/gds if gds > 0 else np.nan
        })

matched = pd.DataFrame(matched_rows)
display(matched.round(4))
matched.to_csv(OUT_TABLES / "scl08_nmos_matched_bias_parameters.csv", index=False)

## Interpretation notes

- `gm` is estimated from the transfer curve using a numerical derivative.
- `gds` is estimated from the output curve using a numerical derivative.
- `ro = 1/gds`; intrinsic gain is estimated as `gm/gds`.
- Near cutoff, ratios such as `gm/ID` are numerically sensitive and should not be treated as reliable design points.
- The `gm` and `ID` pairing assumes matching geometry, bias, and model setup across transfer and output sweeps. Confirm each netlist before using the table for circuit design.